In [1]:
print("ram ram")

ram ram


In [2]:
from pathlib import Path

input_folder = Path("input")   # change path if needed

# Get all files (ignores subfolders)
files = [f for f in input_folder.iterdir() if f.is_file()]

# Print them
for f in files:
    print(f)

input\C Ring pressing NG - Yokoten management reflection.pptx
input\Teeth Surface Forming Machine.pdf
input\Worm Shaft Crack Updated QIP (003).pptx


In [5]:
import os
import uuid
from pathlib import Path
import fitz  # PyMuPDF

# ========== CONFIG ==========
INPUT_DIR = Path("input")          # folder containing PDFs
OUTPUT_TEXT = Path("output/text")
OUTPUT_IMAGES = Path("output/images")
# ============================

# Create output folders
OUTPUT_TEXT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)

# Get all PDF files
pdf_files = list(INPUT_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in input folder.")
else:
    print(f"Found {len(pdf_files)} PDF(s)\n")

for pdf_path in pdf_files:
    pdf_name = pdf_path.stem  # filename without .pdf
    print(f"Processing: {pdf_path.name}")

    # Open PDF
    doc = fitz.open(pdf_path)

    # ---------- 1. Extract Text ----------
    full_text = ""
    for page in doc:
        full_text += page.get_text() + "\n\n"

    text_file = OUTPUT_TEXT / f"{pdf_name}.txt"
    text_file.write_text(full_text.strip(), encoding="utf-8")
    print(f"  → Text saved: {text_file}")

    # ---------- 2. Extract Images ----------
    image_count = 0
    for page_num, page in enumerate(doc, start=1):
        images = page.get_images(full=True)

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]  # usually png or jpeg

            # Unique code
            unique_code = uuid.uuid4().hex[:8]

            # Naming: pdf-name-unique-code.png
            image_filename = f"{pdf_name}-{unique_code}.{image_ext}"
            image_path = OUTPUT_IMAGES / image_filename

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_count += 1

    print(f"  → Extracted {image_count} image(s)\n")
    doc.close()

print("Done!")

Found 1 PDF(s)

Processing: Teeth Surface Forming Machine.pdf
  → Text saved: output\text\Teeth Surface Forming Machine.txt
  → Extracted 23 image(s)

Done!


In [11]:
from PIL import Image
import io
import base64
from pathlib import Path
import ollama
import json

# ================== CONFIG ==================
IMAGE_PATH = Path(r"C:\Users\Administrator\Desktop\okf system\docs-processing-ollama\output\images\Teeth Surface Forming Machine-0b5cf7e8.jpeg")
MODEL = "qwen2.5vl:3b"
MAX_SIZE = 768          # Even smaller
# ============================================

def encode_image_resized(image_path: Path, max_size: int = 768) -> str:
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=80)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

SYSTEM_PROMPT = """
You are an expert industrial document image analyst specialized in manufacturing equipment, HMI screens, and machine interfaces.

Return ONLY a valid JSON object following this exact schema:

{
  "asset_type": "embedded_document_image",
  "unique_image_id": "string",
  "source_file_origin": "string",
  "page_or_sheet_location": "string",
  "component_name": "string",
  "keywords": ["list", "of", "keywords"],
  "visual_specifications": {
    "expected_color": "detailed color description",
    "critical_alignment_angles": "layout description",
    "surface_texture_purity": "interface description"
  },
  "detailed_explanation": "Very detailed explanation of the image",
  "common_defects_and_failures": [
    {
      "defect_type": "string",
      "visual_signs": "string",
      "criticality_level": "HIGH / MEDIUM / LOW"
    }
  ],
  "local_storage_path": "string"
}

Rules:
- Return ONLY pure JSON. No markdown, no extra text.
"""

def analyze_image(image_path: Path):
    print(f"Analyzing: {image_path.name}")
    
    image_b64 = encode_image_resized(image_path, MAX_SIZE)

    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Analyze this industrial HMI/machine image and return the JSON.",
                "images": [image_b64]
            }
        ],
        format="json",
        options={
            "temperature": 0.1,
            "num_ctx": 2048,
            "num_gpu": 0          # ← Force CPU (important)
        }
    )

    try:
        return json.loads(response["message"]["content"])
    except Exception as e:
        print("JSON parse error:", e)
        print("Raw response:", response["message"]["content"])
        return None

if __name__ == "__main__":
    result = analyze_image(IMAGE_PATH)

    if result:
        print(json.dumps(result, indent=2, ensure_ascii=False))
        
        output_file = IMAGE_PATH.with_suffix(".json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        print(f"\nSaved → {output_file}")

Analyzing: Teeth Surface Forming Machine-0b5cf7e8.jpeg
{
  "asset_type": "embedded_document_image",
  "unique_image_id": "1234567890abcdef",
  "source_file_origin": "HMI Fault Log",
  "page_or_sheet_location": "Fault Occurrence Counts Page",
  "component_name": "Fault History Display",
  "keywords": [
    "PLC fault",
    "P2 Link No. 2 Communication Error"
  ],
  "visual_specifications": {
    "expected_color": "Black background with white text and red highlights for important information.",
    "critical_alignment_angles": "The table is centered, with rows aligned horizontally.",
    "surface_texture_purity": "Clean and clear interface without any visible scratches or smudges."
  },
  "detailed_explanation": "This image shows a fault history display on an industrial HMI. It lists two occurrences of faults: one related to the PLC (Programmable Logic Controller) and another involving P2 Link No. 2 communication error with CPU-L1. The date and time are displayed at the top, indicating w

In [12]:
import os
import uuid
import json
import base64
import io
from pathlib import Path
from PIL import Image
import fitz  # PyMuPDF
import ollama

# ====================== CONFIG ======================
INPUT_DIR = Path("input")
OUTPUT_TEXT = Path("output/text")
OUTPUT_IMAGES = Path("output/images")
OUTPUT_FINAL_MD = Path("output/final-md")

MODEL = "qwen2.5vl:3b"          # Use 3b for stability
MAX_IMAGE_SIZE = 768            # Safe size for your GPU
USE_CPU = True                  # Set False if you want to try GPU again
# ====================================================

# Create folders
OUTPUT_TEXT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_FINAL_MD.mkdir(parents=True, exist_ok=True)

def encode_image_resized(image_path: Path, max_size: int = 768) -> str:
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

def analyze_image_with_ollama(image_path: Path) -> dict:
    """Send image to Ollama and get structured JSON"""
    print(f"    → Analyzing image: {image_path.name}")

    image_b64 = encode_image_resized(image_path, MAX_IMAGE_SIZE)

    system_prompt = """
You are an expert industrial document image analyst specialized in manufacturing equipment, HMI screens, and machine interfaces.

Return ONLY a valid JSON object following this exact schema:

{
  "asset_type": "embedded_document_image",
  "unique_image_id": "string",
  "source_file_origin": "string",
  "page_or_sheet_location": "string",
  "component_name": "string",
  "keywords": ["list", "of", "keywords"],
  "visual_specifications": {
    "expected_color": "detailed color description",
    "critical_alignment_angles": "layout description",
    "surface_texture_purity": "interface description"
  },
  "detailed_explanation": "Very detailed explanation of the image",
  "common_defects_and_failures": [
    {
      "defect_type": "string",
      "visual_signs": "string",
      "criticality_level": "HIGH / MEDIUM / LOW"
    }
  ],
  "local_storage_path": "string"
}

Rules:
- Return ONLY pure JSON. No markdown, no extra text.
- Be detailed and technical.
"""

    options = {
        "temperature": 0.1,
        "num_ctx": 2048,
    }
    if USE_CPU:
        options["num_gpu"] = 0

    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": "Analyze this industrial HMI/machine image and return the JSON.",
                    "images": [image_b64]
                }
            ],
            format="json",
            options=options
        )
        return json.loads(response["message"]["content"])
    except Exception as e:
        print(f"    [ERROR] Vision analysis failed: {e}")
        return {
            "error": str(e),
            "local_storage_path": str(image_path)
        }

# ====================== MAIN ======================
pdf_files = list(INPUT_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in input folder.")
else:
    print(f"Found {len(pdf_files)} PDF(s)\n")

for pdf_path in pdf_files:
    pdf_name = pdf_path.stem
    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")

    doc = fitz.open(pdf_path)

    # Create image subfolder for this PDF
    pdf_image_dir = OUTPUT_IMAGES / pdf_name
    pdf_image_dir.mkdir(parents=True, exist_ok=True)

    md_content = []
    md_content.append(f"# {pdf_name}\n")

    for page_num, page in enumerate(doc, start=1):
        md_content.append(f"\n## Page {page_num}\n")

        # ----- Extract Text -----
        text = page.get_text("text").strip()
        if text:
            md_content.append(text)
            md_content.append("\n")

        # ----- Extract Images + Analyze -----
        images = page.get_images(full=True)

        for img_index, img in enumerate(images, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            unique_code = uuid.uuid4().hex[:8]
            image_filename = f"{unique_code}.{image_ext}"
            image_path = pdf_image_dir / image_filename

            # Save image
            with open(image_path, "wb") as f:
                f.write(image_bytes)

            print(f"  → Saved image: {image_path}")

            # Analyze with Ollama
            analysis = analyze_image_with_ollama(image_path)

            # Force correct local path in JSON
            analysis["local_storage_path"] = str(image_path).replace("\\", "/")

            # Insert JSON into Markdown
            md_content.append("\n### Extracted Image Analysis\n")
            md_content.append("```json")
            md_content.append(json.dumps(analysis, indent=2, ensure_ascii=False))
            md_content.append("```\n")

    doc.close()

    # ----- Save intermediate text -----
    text_file = OUTPUT_TEXT / f"{pdf_name}.md"
    text_file.write_text("\n".join(md_content), encoding="utf-8")
    print(f"  → Intermediate MD saved: {text_file}")

    # ----- Save Final MD -----
    final_md_file = OUTPUT_FINAL_MD / f"{pdf_name}.md"
    final_md_file.write_text("\n".join(md_content), encoding="utf-8")
    print(f"  → Final MD saved: {final_md_file}")

print("\nAll done!")

Found 1 PDF(s)


Processing: Teeth Surface Forming Machine.pdf
  → Saved image: output\images\Teeth Surface Forming Machine\00fbdd72.jpeg
    → Analyzing image: 00fbdd72.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\add02362.jpeg
    → Analyzing image: add02362.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\a728b71b.jpeg
    → Analyzing image: a728b71b.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\38ff3699.jpeg
    → Analyzing image: 38ff3699.jpeg
    [ERROR] Vision analysis failed: Expecting ',' delimiter: line 14 column 160 (char 1639)
  → Saved image: output\images\Teeth Surface Forming Machine\5a8833d4.jpeg
    → Analyzing image: 5a8833d4.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\a6b1aec3.jpeg
    → Analyzing image: a6b1aec3.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\1309055c.jpeg
    → Analyzing image: 1309055c.jpeg
  → Saved image: output\images\Teeth Surface Forming Machine\

In [1]:
import os
import uuid
import json
import base64
import io
from pathlib import Path
from PIL import Image
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
import ollama

# ====================== CONFIG ======================
INPUT_DIR = Path("input")
OUTPUT_TEXT = Path("output/text")
OUTPUT_IMAGES = Path("output/images")
OUTPUT_FINAL_MD = Path("output/final-md")

MODEL = "qwen2.5vl:3b"
MAX_IMAGE_SIZE = 768
USE_CPU = True                  # Keep True for stability on your GPU
# ====================================================

# Create folders
OUTPUT_TEXT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_FINAL_MD.mkdir(parents=True, exist_ok=True)

def encode_image_resized(image_path: Path, max_size: int = 768) -> str:
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

def analyze_image_with_ollama(image_path: Path) -> dict:
    print(f"    → Analyzing image: {image_path.name}")

    image_b64 = encode_image_resized(image_path, MAX_IMAGE_SIZE)

    system_prompt = """
You are an expert industrial document image analyst specialized in manufacturing equipment, HMI screens, and machine interfaces.

Return ONLY a valid JSON object following this exact schema:

{
  "asset_type": "embedded_document_image",
  "unique_image_id": "string",
  "source_file_origin": "string",
  "page_or_sheet_location": "string",
  "component_name": "string",
  "keywords": ["list", "of", "keywords"],
  "visual_specifications": {
    "expected_color": "detailed color description",
    "critical_alignment_angles": "layout description",
    "surface_texture_purity": "interface description"
  },
  "detailed_explanation": "Very detailed explanation of the image",
  "common_defects_and_failures": [
    {
      "defect_type": "string",
      "visual_signs": "string",
      "criticality_level": "HIGH / MEDIUM / LOW"
    }
  ],
  "local_storage_path": "string"
}

Rules:
- Return ONLY pure JSON. No markdown, no extra text.
- Be detailed and technical.
"""

    options = {
        "temperature": 0.1,
        "num_ctx": 2048,
    }
    if USE_CPU:
        options["num_gpu"] = 0

    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": "Analyze this industrial HMI/machine image and return the JSON.",
                    "images": [image_b64]
                }
            ],
            format="json",
            options=options
        )
        return json.loads(response["message"]["content"])
    except Exception as e:
        print(f"    [ERROR] Vision analysis failed: {e}")
        return {
            "error": str(e),
            "local_storage_path": str(image_path)
        }

def extract_text_from_shape(shape):
    """Recursively extract text from shapes (including groups)"""
    text = ""
    if shape.has_text_frame:
        for paragraph in shape.text_frame.paragraphs:
            for run in paragraph.runs:
                text += run.text
            text += "\n"
    return text.strip()

# ====================== MAIN ======================
ppt_files = list(INPUT_DIR.glob("*.pptx")) + list(INPUT_DIR.glob("*.ppt"))

if not ppt_files:
    print("No PowerPoint files (.pptx / .ppt) found in input folder.")
else:
    print(f"Found {len(ppt_files)} PowerPoint file(s)\n")

for ppt_path in ppt_files:
    ppt_name = ppt_path.stem
    print(f"\n{'='*60}")
    print(f"Processing: {ppt_path.name}")
    print(f"{'='*60}")

    # Note: python-pptx only supports .pptx well
    if ppt_path.suffix.lower() == ".ppt":
        print("  [WARNING] Old .ppt format is not fully supported. Please convert to .pptx")
        continue

    prs = Presentation(ppt_path)

    # Create image subfolder for this PPT
    ppt_image_dir = OUTPUT_IMAGES / ppt_name
    ppt_image_dir.mkdir(parents=True, exist_ok=True)

    md_content = []
    md_content.append(f"# {ppt_name}\n")

    for slide_num, slide in enumerate(prs.slides, start=1):
        md_content.append(f"\n## Slide {slide_num}\n")

        # ----- Extract Text -----
        slide_text = []
        for shape in slide.shapes:
            text = extract_text_from_shape(shape)
            if text:
                slide_text.append(text)

        if slide_text:
            md_content.append("\n".join(slide_text))
            md_content.append("\n")

        # ----- Extract Images + Analyze -----
        for shape in slide.shapes:
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                try:
                    image = shape.image
                    image_bytes = image.blob
                    image_ext = image.ext  # e.g. 'png', 'jpg'

                    unique_code = uuid.uuid4().hex[:8]
                    image_filename = f"{unique_code}.{image_ext}"
                    image_path = ppt_image_dir / image_filename

                    # Save image
                    with open(image_path, "wb") as f:
                        f.write(image_bytes)

                    print(f"  → Saved image: {image_path}")

                    # Analyze with Ollama
                    analysis = analyze_image_with_ollama(image_path)
                    analysis["local_storage_path"] = str(image_path).replace("\\", "/")
                    analysis["page_or_sheet_location"] = f"Slide {slide_num}"

                    # Insert JSON into Markdown
                    md_content.append("\n### Extracted Image Analysis\n")
                    md_content.append("```json")
                    md_content.append(json.dumps(analysis, indent=2, ensure_ascii=False))
                    md_content.append("```\n")

                except Exception as e:
                    print(f"  [ERROR] Failed to extract image: {e}")

    # ----- Save intermediate text -----
    text_file = OUTPUT_TEXT / f"{ppt_name}.md"
    text_file.write_text("\n".join(md_content), encoding="utf-8")
    print(f"  → Intermediate MD saved: {text_file}")

    # ----- Save Final MD -----
    final_md_file = OUTPUT_FINAL_MD / f"{ppt_name}.md"
    final_md_file.write_text("\n".join(md_content), encoding="utf-8")
    print(f"  → Final MD saved: {final_md_file}")

print("\nAll PowerPoint files processed!")

Found 2 PowerPoint file(s)


Processing: C Ring pressing NG - Yokoten management reflection.pptx
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\453cc8fb.png
    → Analyzing image: 453cc8fb.png
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\607822a1.jpg
    → Analyzing image: 607822a1.jpg
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\c1a8837b.png
    → Analyzing image: c1a8837b.png
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\a289bd24.jpg
    → Analyzing image: a289bd24.jpg
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\19332e81.png
    → Analyzing image: 19332e81.png
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\e2b2c3b4.png
    → Analyzing image: e2b2c3b4.png
  → Saved image: output\images\C Ring pressing NG - Yokoten management reflection\07eb796e.jpg
    → Analyzing image: